# ▶️ Run this in Google Colab — start here

This is the Colab-ready version of the Sydney PM2.5 forecasting notebook.
**Run the next three cells in order, then use *Runtime → Run all*.**

1. **Setup** — installs the stats library and writes a fallback sample file.
2. **Upload your data** *(optional)* — click the button and choose your `sydney_pm25.csv`.
   Skip it and the notebook runs on a built-in synthetic sample so you can see it work immediately.
3. The analysis then runs unchanged.

> Your CSV needs exactly two columns: `date` and `pm25` (daily-average µg/m³) — the format `fetch_data.py` produces from the real OpenAQ API.

---

In [ ]:
# === SETUP (run first) ===
# Install the time-series library (Colab usually has it, --upgrade to be safe).
!pip install statsmodels --upgrade --quiet

# Write a built-in synthetic sample to disk as a fallback, so 'Run all' works
# even before you upload anything. Your upload (next cell) overrides this.
import base64
_SAMPLE_B64 = "..."
with open('sydney_pm25.csv', 'wb') as _f:
    _f.write(base64.b64decode(_SAMPLE_B64))

CSV_PATH = 'sydney_pm25.csv'  # default; the upload cell may change this
print('Setup complete. A synthetic sample is ready as a fallback.')
print('Upload your real CSV in the next cell, or just keep going to use the sample.')

In [ ]:
# === UPLOAD YOUR DATA (optional) ===
# Click 'Choose Files' and select your sydney_pm25.csv (real data from
# fetch_data.py, or the synthetic sample from the repo's data/ folder).
# Skip this cell to use the built-in sample instead.
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        CSV_PATH = list(uploaded.keys())[0]
        print(f'Using uploaded file: {CSV_PATH}')
    else:
        print('No file chosen — using the built-in synthetic sample.')
except ImportError:
    print('Not running in Colab — using local CSV_PATH if set.')

# Forecasting Sydney's PM2.5 Air Pollution — A Day-Ahead Early-Warning System

**Author:** Ragul Balaji Selvaraj · Melbourne, Australia
**Contact:** ragulbalajiselvaraj@gmail.com · [LinkedIn](https://www.linkedin.com/in/ragulbalajiselvaraj) · [GitHub](https://github.com/ragul-selvaraj)

---

## The question

> **Can we forecast Sydney's PM2.5 air pollution one day ahead accurately enough to flag hazardous-air days *before* they happen?**

PM2.5 — fine particulate matter smaller than 2.5 microns — is the air pollutant most strongly linked to harm to human health. It lodges deep in the lungs and bloodstream. If a health agency could reliably know *tomorrow's* air quality *today*, it could issue early warnings so that vulnerable people (asthmatics, the elderly, outdoor workers) can take precautions.

This notebook treats that as a concrete forecasting problem and answers it honestly:

1. **Understand** the data — Sydney's daily PM2.5, including the 2019–20 "Black Summer" bushfire smoke episode.
2. **Establish a baseline** any useful model must beat (the "tomorrow = today" naïve forecast).
3. **Diagnose** the series properly — trend, seasonality, stationarity (ADF test), autocorrelation (ACF/PACF).
4. **Model** with ARIMA/SARIMA, validated by **walk-forward validation** so there is *no data leakage*.
5. **Reframe as early warning** — turn the forecast into a yes/no "hazardous tomorrow?" classifier and measure how many dangerous days we'd actually catch.
6. **Recommend** something a health agency could act on.

## A note on the data

The data is **real**, sourced from the [OpenAQ](https://openaq.org) v3 API, which aggregates the NSW Government's air-quality monitoring network — the densest and longest-running in Australia. The companion script `fetch_data.py` downloads it into `data/sydney_pm25.csv`. This notebook reads only that CSV, so the analysis is fully reproducible and runs without a network connection. See the README for the exact fetch command.


## 0. Setup

A small, deliberately classic toolkit — `pandas` for data handling, `matplotlib`/`seaborn` for charts, and `statsmodels` for the time-series statistics. No deep learning, no heavy framework: for a single-city day-ahead forecast, classical methods are the right tool, and restraint is itself a signal of judgment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

import warnings
warnings.filterwarnings("ignore")   # statsmodels is chatty during walk-forward fits

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 110

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Setup complete.")

## 1. Load and inspect the data

The CSV has just two columns — `date` and `pm25` (the daily-average concentration in µg/m³). We parse `date` into a real datetime, sort chronologically, and set it as the index. **For time series, a proper `DatetimeIndex` is essential**: it's what lets pandas resample, fill gaps by calendar date, and lets statsmodels understand the spacing between observations.

In [ ]:
# --- Data loading (Colab-ready) ---
# This cell uses the CSV you uploaded in the setup cell above.
# `CSV_PATH` was set there. If you skipped the upload, it falls back to the
# synthetic sample that ships with this notebook so 'Run all' still works.
import os

csv_path = globals().get('CSV_PATH', None)
if not csv_path or not os.path.exists(csv_path):
    # fallback: synthetic sample written by the setup cell
    csv_path = 'sydney_pm25.csv'

df = pd.read_csv(csv_path, parse_dates=['date'])
df = df.sort_values('date').set_index('date')

print(f'Loaded: {csv_path}')
print(f'Rows: {len(df)}')
print(f'Date range: {df.index.min().date()} -> {df.index.max().date()}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Quick statistical portrait of the PM2.5 series.
df["pm25"].describe().round(2)

**Reading the summary.** The mean sits in the low double digits (µg/m³), typical for a temperate coastal city. But watch the **max** versus the **75th percentile**: the maximum is many times larger than the bulk of the distribution. That long right tail is the bushfire smoke — exactly the kind of dangerous, rare event an early-warning system exists to catch. We'll keep coming back to it.

## 2. Make the time series regular (and handle gaps)

Real sensor data has gaps — stations go offline for maintenance, or a day's coverage falls below the quality threshold and gets dropped. A time-series model needs observations on a **regular daily grid** with no missing dates. So we:

1. **Reindex** onto a complete daily calendar (this *inserts* `NaN` rows for any missing days, making the gaps explicit).
2. **Interpolate** those gaps. We use time-based linear interpolation — for a short gap, yesterday-and-tomorrow is a reasonable estimate of a missing day. We also limit how many consecutive days we're willing to fill, so we never invent a long stretch of data.

In [ ]:
full_index = pd.date_range(df.index.min(), df.index.max(), freq="D")
missing_before = full_index.difference(df.index)
print(f"Calendar days in range : {len(full_index)}")
print(f"Days actually present  : {len(df)}")
print(f"Missing days to fill   : {len(missing_before)}")

ts = df["pm25"].reindex(full_index)
ts = ts.interpolate(method="time", limit=7, limit_direction="both")
ts = ts.dropna()   # drop any gap longer than the 7-day fill limit
ts.index.name = "date"

print(f"\nFinal continuous series: {len(ts)} days, "
      f"{ts.index.min().date()} -> {ts.index.max().date()}")
print(f"Any remaining missing?  {ts.isna().any()}")

## 3. Exploratory analysis — what does Sydney's air actually do?

### 3.1 The full daily series

Before any modelling, look at the data. The single most important plot in the whole project is just PM2.5 over time.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(ts.index, ts.values, lw=0.8, color="#2b6cb0")
ax.set_title("Sydney daily PM2.5 (µg/m³)", fontsize=13, weight="bold")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.set_xlabel("")

# WHO 24-hour guideline (2021): 15 µg/m³. A useful health reference line.
ax.axhline(15, color="#dd6b20", ls="--", lw=1, label="WHO 24h guideline (15 µg/m³)")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Two things jump out:

- **A regular wave** — air is consistently worse in the cooler months. In Sydney, winter brings wood-heater smoke and more "still" days where pollution doesn't disperse.
- **One enormous spike** — late 2019 into early 2020. That is the **Black Summer bushfire crisis**, when smoke blanketed Sydney for weeks and PM2.5 reached levels rarely seen in a developed-world city. This is the Australian story at the heart of the project: the early-warning question matters *most* precisely on days like these.

### 3.2 Seasonality, more directly

Averaging PM2.5 by calendar month makes the winter pattern unmistakable.

In [ ]:
by_month = ts.groupby(ts.index.month).mean()
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(11, 4.5))
bars = ax.bar(month_names, by_month.values, color="#4a5568")
# highlight the winter (austral) months
for i, b in enumerate(bars):
    if i in (4,5,6,7):     # May-Aug
        b.set_color("#2b6cb0")
ax.set_title("Average PM2.5 by month — winter (blue) is worst", weight="bold")
ax.set_ylabel("Mean PM2.5 (µg/m³)")
plt.tight_layout()
plt.show()

### 3.3 Classical decomposition

`statsmodels`' `seasonal_decompose` splits the series into three parts: a slow **trend**, a repeating **seasonal** cycle, and the leftover **residual**. We use an additive model with a 365-day period (annual seasonality). This is a descriptive tool — it helps us *see* the structure we'll later ask the model to capture.

In [ ]:
decomp = seasonal_decompose(ts, model="additive", period=365)

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
decomp.observed.plot(ax=axes[0], color="#2b6cb0"); axes[0].set_ylabel("Observed")
decomp.trend.plot(ax=axes[1], color="#2f855a");    axes[1].set_ylabel("Trend")
decomp.seasonal.plot(ax=axes[2], color="#dd6b20"); axes[2].set_ylabel("Seasonal")
decomp.resid.plot(ax=axes[3], color="#718096", lw=0.6); axes[3].set_ylabel("Residual")
axes[0].set_title("Additive decomposition of Sydney PM2.5", weight="bold")
axes[3].set_xlabel("")
plt.tight_layout()
plt.show()

The **seasonal** panel confirms a clean annual cycle. The **residual** panel is where the bushfire spike lands — decomposition treats it as a huge "leftover", which is the correct reading: a bushfire is not a regular seasonal event, it's an extreme shock. That distinction matters for what follows.

## 4. Is the series stationary? (The ADF test)

ARIMA models assume the series is **stationary** — that its statistical properties (mean, variance) don't drift over time. Many real series aren't, and we have to check rather than assume.

The standard tool is the **Augmented Dickey-Fuller (ADF) test**. Its logic:

- **Null hypothesis (H₀):** the series is *non-stationary* (has a "unit root").
- If the **p-value is small** (conventionally < 0.05), we reject H₀ and conclude the series is *stationary*.

Let's run it on the raw series.

In [ ]:
def run_adf(series, label):
    stat, pval, lags, nobs, crit, _ = adfuller(series.dropna(), autolag="AIC")
    print(f"ADF test — {label}")
    print(f"  ADF statistic : {stat:.4f}")
    print(f"  p-value       : {pval:.4f}")
    print(f"  lags used     : {lags}")
    verdict = "STATIONARY (reject H0)" if pval < 0.05 else "NON-stationary (cannot reject H0)"
    print(f"  -> {verdict}\n")
    return pval

p_raw = run_adf(ts, "raw PM2.5")

If the raw series comes back non-stationary, the standard fix is **differencing** — modelling the day-to-day *change* instead of the level. This is the "I" (Integrated) in ARIMA, the `d` parameter. One round of differencing usually does it.

In [ ]:
ts_diff = ts.diff().dropna()
p_diff = run_adf(ts_diff, "first-differenced PM2.5")

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ts.plot(ax=axes[0], color="#2b6cb0", lw=0.8); axes[0].set_title("Original", weight="bold"); axes[0].set_ylabel("PM2.5")
ts_diff.plot(ax=axes[1], color="#805ad5", lw=0.8); axes[1].set_title("First difference (day-to-day change)", weight="bold"); axes[1].set_ylabel("Δ PM2.5")
plt.tight_layout(); plt.show()

The differenced series hovers around a constant mean of zero — visually stationary, and the ADF p-value confirms it. So we'll use **d = 1** in the ARIMA model.

## 5. Choosing AR and MA terms — ACF and PACF

With `d` settled, we need the other two ARIMA parameters:

- **`p`** — the **AutoRegressive** order: how many *past values* to use.
- **`q`** — the **Moving Average** order: how many *past forecast errors* to use.

The classic way to read these off is two plots, run on the **differenced** series:

- **PACF** (Partial AutoCorrelation) suggests **`p`** — look for where it cuts off.
- **ACF** (AutoCorrelation) suggests **`q`** — likewise.

These are guides, not gospel; we'll confirm the final choice with a proper validation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(ts_diff, ax=axes[0], lags=30, color="#2b6cb0")
axes[0].set_title("ACF of differenced series (suggests q)", weight="bold")
plot_pacf(ts_diff, ax=axes[1], lags=30, method="ywm", color="#2f855a")
axes[1].set_title("PACF of differenced series (suggests p)", weight="bold")
plt.tight_layout(); plt.show()

Both plots show their strongest spike at the first lag or two and then drop off quickly into the shaded confidence band. That points to **small** `p` and `q` (around 1–2) — pleasingly simple. A sprawling, high-order model would be a red flag for over-fitting. We'll carry a shortlist of small candidate orders into validation.

## 6. The honest test — baseline first, then walk-forward validation

This is the methodological heart of the project, and the part most amateur forecasting projects get wrong.

### 6.1 The baseline you have to beat

Before celebrating any model, you need a **baseline** — the dumbest reasonable forecast. For daily PM2.5 the natural one is the **seasonal naïve / persistence** forecast:

> **"Tomorrow's PM2.5 will equal today's."**

This is a genuinely strong baseline because air quality is sticky day-to-day. **A model that can't beat persistence is worthless**, no matter how sophisticated it looks. Stating the baseline up front keeps us honest.

### 6.2 Why walk-forward validation (no data leakage)

You cannot evaluate a forecaster by training it on the whole dataset and scoring it on data it already saw — that **leaks** future information and produces a fantasy accuracy number.

Instead we use **walk-forward validation**, which mimics real life:

1. Train on data up to day *t*.
2. Forecast day *t+1* (which the model has never seen).
3. Record the error.
4. Add the *real* day *t+1* to the training set, slide forward, repeat.

At no point does the model see the future. This is the single most important thing separating a trustworthy forecasting project from a misleading one.

In [ ]:
# Hold out the final ~6 months as the walk-forward test window. A longer
# window is a more demanding test and ensures we evaluate across a full winter
# (when hazardous days actually occur), not just a quiet stretch.
TEST_DAYS = 180
train_full = ts.iloc[:-TEST_DAYS]
test = ts.iloc[-TEST_DAYS:]
print(f"Training history : {len(train_full)} days "
      f"({train_full.index.min().date()} -> {train_full.index.max().date()})")
print(f"Walk-forward test: {len(test)} days "
      f"({test.index.min().date()} -> {test.index.max().date()})")

In [ ]:
def mae(a, b):  return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b))**2)))

# ---- Baseline: persistence (tomorrow = today), walk forward ----
baseline_pred, actuals, dates_eval = [], [], []
history = list(train_full.values)
for d, actual in zip(test.index, test.values):
    baseline_pred.append(history[-1])   # yesterday's value
    actuals.append(actual)
    dates_eval.append(d)
    history.append(actual)              # reveal the truth, step forward

baseline_mae  = mae(actuals, baseline_pred)
baseline_rmse = rmse(actuals, baseline_pred)
print(f"BASELINE (persistence)   MAE = {baseline_mae:.3f}   RMSE = {baseline_rmse:.3f}")

Now the ARIMA model, evaluated the **same** way — refitting at every step and forecasting one day ahead. We test a few small candidate orders (from the ACF/PACF read) and keep whichever wins on the walk-forward MAE. Refitting 120 times for several orders takes a moment; that cost is the price of an honest number.

In [ ]:
def walk_forward_arima(train_series, test_series, order):
    preds, hist = [], list(train_series.values)
    for actual in test_series.values:
        model = ARIMA(hist, order=order)
        fit = model.fit()
        yhat = fit.forecast(steps=1)[0]
        preds.append(float(yhat))
        hist.append(float(actual))   # reveal truth before next step
    return preds

candidate_orders = [(1,1,1), (2,1,1), (1,1,2), (2,1,2)]
results = {}
for order in candidate_orders:
    preds = walk_forward_arima(train_full, test, order)
    results[order] = {
        "preds": preds,
        "mae":  mae(actuals, preds),
        "rmse": rmse(actuals, preds),
    }
    print(f"ARIMA{order}   MAE = {results[order]['mae']:.3f}   RMSE = {results[order]['rmse']:.3f}")

best_order = min(results, key=lambda o: results[o]["mae"])
best = results[best_order]
print(f"\nBest model: ARIMA{best_order}  (MAE = {best['mae']:.3f})")

## 7. Did the model beat the baseline?

The only comparison that matters: best ARIMA vs. the persistence baseline, on identical walk-forward terms.

In [ ]:
improvement = (baseline_mae - best["mae"]) / baseline_mae * 100

summary = pd.DataFrame({
    "Model": ["Baseline (persistence)", f"ARIMA{best_order}"],
    "MAE":   [round(baseline_mae, 3),  round(best["mae"], 3)],
    "RMSE":  [round(baseline_rmse, 3), round(best["rmse"], 3)],
})
print(summary.to_string(index=False))
print(f"\nARIMA improves MAE by {improvement:.1f}% vs. the naïve baseline.")
if improvement > 0:
    print("The model earns its place: it beats 'tomorrow = today' on average error.")
else:
    print("The model does NOT beat persistence on this data — an honest null result,")
    print("and an informative one: for a sticky daily series, the naive forecast is a")
    print("genuinely hard baseline to beat. We report it rather than hide it.")

In [ ]:
# Visualise actual vs. both forecasts across the test window.
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(dates_eval, actuals, color="#1a202c", lw=1.6, label="Actual")
ax.plot(dates_eval, baseline_pred, color="#cbd5e0", lw=1.2, ls="--", label="Baseline (persistence)")
ax.plot(dates_eval, best["preds"], color="#2b6cb0", lw=1.3, label=f"ARIMA{best_order}")
ax.set_title("Day-ahead forecasts vs. actual PM2.5 (walk-forward test window)", weight="bold")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
# Where does the error come from? A residual plot for the winning model.
resid = np.asarray(actuals) - np.asarray(best["preds"])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(dates_eval, resid, color="#718096", lw=0.9)
axes[0].axhline(0, color="#1a202c", lw=0.8)
axes[0].set_title("Forecast errors over time", weight="bold")
axes[0].set_ylabel("Actual − Predicted")
axes[0].tick_params(axis="x", rotation=45)
sns.histplot(resid, kde=True, ax=axes[1], color="#2b6cb0")
axes[1].set_title("Distribution of forecast errors", weight="bold")
axes[1].set_xlabel("Error (µg/m³)")
plt.tight_layout(); plt.show()
print(f"Mean error (bias): {resid.mean():.3f}   |   Error std: {resid.std():.3f}")

## 8. From forecast to early warning — the classification view

A health agency doesn't act on a decimal number; it acts on a **decision**: *"Will tomorrow be hazardous — yes or no?"* So we reframe the regression forecast as a **binary classifier**.

We set a health threshold and label each day. Australia's air-quality bands put PM2.5 into "poor / very poor" territory well above the WHO 15 µg/m³ guideline; for a clear early-warning trigger we use **25 µg/m³** as the "hazardous day" line (roughly where sensitive groups are advised to limit outdoor exertion). Then we ask: **on the days the model predicted, how many truly hazardous days would we have caught?**

The right metrics here are **not** plain accuracy (hazardous days are rare, so a lazy "never hazardous" guess scores high). Instead:

- **Recall (sensitivity)** — of all the genuinely hazardous days, what fraction did we flag? *For a health warning, this is the metric that matters most — a missed hazardous day is the costly error.*
- **Precision** — of the days we flagged, what fraction were truly hazardous? (Too many false alarms and people stop listening.)

In [ ]:
HAZARD_THRESHOLD = 25.0   # µg/m³ — the "hazardous day" early-warning trigger

actual_haz   = np.asarray(actuals)      >= HAZARD_THRESHOLD
arima_haz    = np.asarray(best["preds"]) >= HAZARD_THRESHOLD
baseline_haz = np.asarray(baseline_pred) >= HAZARD_THRESHOLD

def classification_report_simple(y_true, y_pred, label):
    tp = int(np.sum(y_true & y_pred))
    fp = int(np.sum(~y_true & y_pred))
    fn = int(np.sum(y_true & ~y_pred))
    tn = int(np.sum(~y_true & ~y_pred))
    precision = tp / (tp + fp) if (tp+fp) else float("nan")
    recall    = tp / (tp + fn) if (tp+fn) else float("nan")
    f1 = (2*precision*recall/(precision+recall)
          if (precision and recall and not np.isnan(precision) and not np.isnan(recall)) else float("nan"))
    print(f"{label}")
    print(f"  Hazardous days correctly flagged (TP): {tp}")
    print(f"  False alarms (FP)                    : {fp}")
    print(f"  MISSED hazardous days (FN)           : {fn}")
    print(f"  Quiet days correctly cleared (TN)    : {tn}")
    print(f"  Precision: {precision:.2f}   Recall: {recall:.2f}   F1: {f1:.2f}\n")
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

n_haz = int(actual_haz.sum())
print(f"Hazardous days in test window (>= {HAZARD_THRESHOLD} µg/m³): "
      f"{n_haz} of {len(actual_haz)}\n")
if n_haz == 0:
    print("No hazardous days fell in this particular test window, so recall is\n"
          "undefined here. Re-run with a longer TEST_DAYS (to span a winter) or a\n"
          "lower HAZARD_THRESHOLD to evaluate early-warning performance. The\n"
          "metrics below will populate once the window contains hazardous days.\n")
m_base  = classification_report_simple(actual_haz, baseline_haz, "BASELINE (persistence) as a warning system")
m_arima = classification_report_simple(actual_haz, arima_haz,    f"ARIMA{best_order} as a warning system")

In [ ]:
# Confusion matrix for the ARIMA early-warning system.
from collections import OrderedDict
cm = np.array([[int(np.sum(~actual_haz & ~arima_haz)), int(np.sum(~actual_haz & arima_haz))],
               [int(np.sum(actual_haz & ~arima_haz)),  int(np.sum(actual_haz & arima_haz))]])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Pred: Safe","Pred: Hazardous"],
            yticklabels=["Actual: Safe","Actual: Hazardous"], ax=ax)
ax.set_title(f"Early-warning confusion matrix — ARIMA{best_order}", weight="bold")
plt.tight_layout(); plt.show()

**How to read this.** The bottom-right cell is the win — hazardous days we successfully predicted a day in advance. The bottom-left cell is the danger — hazardous days we *missed*. An early-warning system lives and dies on keeping that bottom-left number low (high recall), while not crying wolf so often (top-right) that people tune it out.

## 9. Recommendation for a health agency

*(The exact numbers below are printed from the run above so this reads as a real conclusion.)*

In [ ]:
print(chr(61)*70)
print("FINDINGS SUMMARY - Sydney PM2.5 day-ahead forecasting")
print(chr(61)*70)
print()
print("Forecast accuracy (walk-forward, no leakage):")
print(f"  - Naive baseline MAE      : {baseline_mae:.2f} ug/m3")
print(f"  - Best model ARIMA{best_order}  : {best['mae']:.2f} ug/m3")
print(f"  - Improvement over naive  : {improvement:.1f}%")
print()
print(f"Early-warning performance (hazard threshold = {HAZARD_THRESHOLD:.0f} ug/m3):")
print(f"  - Hazardous days in test period   : {int(actual_haz.sum())}")
print(f"  - Caught by ARIMA (recall)        : {m_arima['recall']:.0%}")
print(f"  - False-alarm-adjusted (precision): {m_arima['precision']:.0%}")
print(f"  - Hazardous days MISSED           : {m_arima['fn']}")
print(chr(61)*70)

### What I'd actually tell the agency

**1. A simple day-ahead forecast is genuinely useful — but with a clear caveat.**
On *ordinary* days, even the naïve "tomorrow = today" rule is a decent guide because air quality is sticky day-to-day. ARIMA lands in the same ballpark — sometimes a little better, sometimes not — which is itself the finding: for routine next-day advisories to sensitive groups, a transparent baseline is already a reasonable tool, and a heavier model is not automatically worth its complexity.

**2. Watch out for a subtle trap: the smoother model can be *worse* at catching spikes.**
A model that minimises average error (like ARIMA) tends to "smooth" its forecasts, which can make it *under*-predict sudden jumps — so it may flag *fewer* hazardous days than the blunt persistence rule. Compare the recall of the two systems above. For an early-warning use case, average error is the wrong thing to optimise alone; catching the dangerous days (recall) matters more, and the simplest method sometimes does that better. That is a judgment a recruiter-facing analyst should make explicit rather than bury.

**3. The hard case is exactly the case that matters: extreme bushfire smoke.**
Statistical models forecast the *next* day from recent days. The *onset* of a bushfire smoke event is a sudden shock with little warning in the prior day's value — so the first dangerous day of an episode is the one most likely to be missed by **any** persistence-style or ARIMA model. Once an episode is underway, day-ahead forecasting tracks it well; it's the *first* spike that's hard.

**4. Practical recommendation.**
- Use the day-ahead forecast as a **baseline advisory tool** for routine air-quality warnings — it's cheap, transparent, and beats guessing.
- For **bushfire season, do not rely on it alone.** Pair it with fire-activity and weather inputs (wind direction, fire locations, satellite smoke detection) that carry the *leading* signal a univariate PM2.5 model cannot see. The natural next step for this project is exactly that: add Open-Meteo weather drivers (wind disperses pollution) and test whether they improve early detection of episode *onset*.
- Tune the hazard threshold to the agency's tolerance for false alarms. A lower threshold catches more dangerous days (higher recall) at the cost of more false alarms — that's a public-health value judgment, not a statistical one, and it should be made explicitly.

### Honest limitations

- **Univariate.** This model sees only PM2.5's own past. It has no weather or fire data, which is why episode *onset* is its weak point. This is a deliberate scoping choice for a clean baseline, and the clearest avenue for improvement.
- **One pollutant, one city.** PM2.5 in Sydney. The method generalises, the specific numbers don't.
- **Daily resolution.** Sub-daily spikes (e.g. a smoky evening) are averaged away. For acute warnings, hourly modelling would be the extension.
- **Threshold choice.** The 25 µg/m³ hazard line is a defensible but adjustable convention, not a law of nature.

---

### What this project demonstrates

A complete, honest analyst workflow: a real public-health question; real OpenAQ/NSW data ingested programmatically; proper time-series diagnostics (decomposition, ADF, ACF/PACF); a baseline the model had to beat; **walk-forward validation with no data leakage**; a regression-to-classification reframe aimed at the actual decision; and a recommendation that's candid about where the method works and where it doesn't. The restraint — classical methods, clearly explained — is the point.
